# Advanced pyALF part 1: Running ALF

In [1]:
# Enable Jupyter Widget support for matplotlib
# Needed for check_warmup and check_rebin
%matplotlib widget

In [2]:
import py_alf

# Getting help

I you need help on how to use pyALF, the python built-in function `help()`might be useful. Alternately, the pyALF documentation at [](https://gitpages.physik.uni-wuerzburg.de/Jonas_schwab/pyalf-docu/source/front.html) might also help.

In [3]:
help(py_alf)

Help on package py_alf:

NAME
    py_alf - pyALF, a Python package for the Algorithms for Lattice Fermions (ALF).

PACKAGE CONTENTS
    alf_source
    ana
    analysis
    check_common
    check_rebin_ipy
    check_rebin_tk
    check_warmup_ipy
    check_warmup_tk
    cli (package)
    exceptions
    init_layout
    lattice
    lattices_x86-64
    simulation
    utils

CLASSES
    builtins.object
        py_alf.alf_source.ALF_source
        py_alf.lattice.Lattice
        py_alf.simulation.Simulation
    
    class ALF_source(builtins.object)
     |  ALF_source(alf_dir=None, branch=None, url='https://git.physik.uni-wuerzburg.de/ALF/ALF.git')
     |  
     |  Objet representing ALF source code.
     |  
     |  Parameters
     |  ----------
     |  alf_dir : path-like object, default=os.getenv('ALF_DIR', './ALF')
     |      Directory containing the ALF source code. If the directory does
     |      not exist, the source code will be fetched from a server.
     |      Defaults to environ

In [4]:
from pprint import pprint  # Pretty print
from py_alf import ALF_source, Simulation  # Interface with ALF

## ALF_source

In [5]:
help(ALF_source)

Help on class ALF_source in module py_alf.alf_source:

class ALF_source(builtins.object)
 |  ALF_source(alf_dir=None, branch=None, url='https://git.physik.uni-wuerzburg.de/ALF/ALF.git')
 |  
 |  Objet representing ALF source code.
 |  
 |  Parameters
 |  ----------
 |  alf_dir : path-like object, default=os.getenv('ALF_DIR', './ALF')
 |      Directory containing the ALF source code. If the directory does
 |      not exist, the source code will be fetched from a server.
 |      Defaults to environment variable $ALF_DIR if defined, otherwise
 |      to './ALF'.
 |  branch : str, optional
 |      If specified, this will be checked out by git.
 |  url : str, default='https://git.physik.uni-wuerzburg.de/ALF/ALF.git'
 |      Address from where to clone ALF if alf_dir does not exist.
 |  
 |  Methods defined here:
 |  
 |  __init__(self, alf_dir=None, branch=None, url='https://git.physik.uni-wuerzburg.de/ALF/ALF.git')
 |      Initialize self.  See help(type(self)) for accurate signature.
 |  

Create instance of `ALF_source`, choosing to checkout the git branch `'Nematic_Dirac_demo'`, containing the Hamiltonian we created yesterday.

Reminder: Directory containing the ALF code is taken from environment variable `$ALF_DIR`, if present.

In [5]:
alf_src = ALF_source(
    branch='Nematic_Dirac_demo',
)

Checking out branch Nematic_Dirac_demo
Your branch is up to date with 'origin/Nematic_Dirac_demo'.


Already on 'Nematic_Dirac_demo'


If the branch "Nematic_Dirac_demo" is not known, try the following command

In [7]:
!cd $ALF_DIR && git pull

Enter passphrase for key '/home/jonas/.ssh/id_ed25519': 


Check available Hamiltonians:

In [6]:
alf_src.get_ham_names()

['Kondo',
 'Hubbard',
 'Hubbard_Plain_Vanilla',
 'tV',
 'LRC',
 'Z2_Matter',
 'Spin_Peierls',
 'Nematic_Dirac_demo']

Print valid parameters in their defaults for Nematic Dirac Hamiltonian, including QMC parameters:

In [7]:
pprint(alf_src.get_default_params('Nematic_Dirac_demo'))

OrderedDict([('VAR_Nematic_Dirac',
              {'Dtau': {'comment': 'Imaginary time step size',
                        'defined_in_base': False,
                        'value': 0.1},
               'Ham_J': {'comment': 'Ferromagnetic Ising interaction',
                         'defined_in_base': False,
                         'value': 1.0},
               'Ham_chem': {'comment': 'Chemical potential',
                            'defined_in_base': False,
                            'value': 0.0},
               'Ham_h': {'comment': 'Ising transverse field',
                         'defined_in_base': False,
                         'value': 3.0},
               'Ham_xi': {'comment': 'Coupling Ising spins <-> fermions',
                          'defined_in_base': False,
                          'value': 1.0},
               'L1': {'comment': 'Size of lattice in a1 direction',
                      'defined_in_base': False,
                      'value': 4},
               'L2': {

## Simulation object

In [10]:
help(Simulation)

Help on class Simulation in module py_alf.simulation:

class Simulation(builtins.object)
 |  Simulation(alf_src, ham_name, sim_dict, **kwargs)
 |  
 |  Object corresponding to an ALF simulation.
 |  
 |  Parameters
 |  ----------
 |  alf_src : ALF_source
 |      Objet representing ALF source code.
 |  ham_name : str
 |      Name of the Hamiltonian.
 |  sim_dict : dict or list of dicts
 |      Dictionary specfying parameters owerwriting defaults.
 |      Can be a list of dictionaries to enable parallel tempering.
 |  sim_dir : path-like object, optional
 |      Directory in which the Monte Carlo will be run.
 |      If not specified, sim_dir is generated from sim_dict.
 |  sim_root : path-like object, default="ALF_data"
 |      Directory to prepend to sim_dir.
 |  mpi : bool, default=False
 |      Employ MPI.
 |  parallel_params : bool, default=False
 |      Run independent parameter sets in parallel.
 |      Based on parallel tempering, but without exchange steps.
 |  n_mpi : int, defa

## Run a simulation

In [8]:
sim = Simulation(
    alf_src,
    'Nematic_Dirac_demo',
    {
        # Model specific parameters
        'L1': 4,
        'L2': 4,
        'beta': 4.,
        'Ham_xi': 0.25,
        'Ham_h': 3,
        # QMC parameters
        'Ltau': 1,
        'CPU_MAX': .01,
        'NSweep': 20,
    },
    machine='gnu',
    mpi=True,
    n_mpi=8,
)

In [12]:
sim.get_directories()

['/home/jonas/Downloads/ALF_2024-advanced_pyALF/ALF_data/Nematic_Dirac_demo_L1=4_L2=4_beta=4.0_xi=0.25_h=3']

In [13]:
sim.compile()

Checking out branch Nematic_Dirac_demo
Your branch is up to date with 'origin/Nematic_Dirac_demo'.


Already on 'Nematic_Dirac_demo'


Compiling ALF... 
Cleaning up Prog/
Cleaning up Libraries/
Cleaning up Analysis/
Compiling Libraries


ar: creating modules_90.a
ar: creating libqrref.a


Compiling Analysis
Compiling Program
Parsing Hamiltonian parameters
filenames: Hamiltonians/Hamiltonian_Kondo_smod.F90 Hamiltonians/Hamiltonian_Kondo_read_write_parameters.F90
filenames: Hamiltonians/Hamiltonian_Hubbard_smod.F90 Hamiltonians/Hamiltonian_Hubbard_read_write_parameters.F90
filenames: Hamiltonians/Hamiltonian_Hubbard_Plain_Vanilla_smod.F90 Hamiltonians/Hamiltonian_Hubbard_Plain_Vanilla_read_write_parameters.F90
filenames: Hamiltonians/Hamiltonian_tV_smod.F90 Hamiltonians/Hamiltonian_tV_read_write_parameters.F90
filenames: Hamiltonians/Hamiltonian_LRC_smod.F90 Hamiltonians/Hamiltonian_LRC_read_write_parameters.F90
filenames: Hamiltonians/Hamiltonian_Z2_Matter_smod.F90 Hamiltonians/Hamiltonian_Z2_Matter_read_write_parameters.F90
filenames: Hamiltonians/Hamiltonian_Spin_Peierls_smod.F90 Hamiltonians/Hamiltonian_Spin_Peierls_read_write_parameters.F90
filenames: Hamiltonians/Hamiltonian_Nematic_Dirac_demo_smod.F90 Hamiltonians/Hamiltonian_Nematic_Dirac_demo_read_write_parameter

`Simulation` generates the directory name the simulation is run in out of the non-default parameters, prepended by "ALF_data". For the first simulation, this is:

In [14]:
sim.run()

Prepare directory "/home/jonas/Downloads/ALF_2024-advanced_pyALF/ALF_data/Nematic_Dirac_demo_L1=4_L2=4_beta=4.0_xi=0.25_h=3" for Monte Carlo run.
Resuming previous run.
Run /home/jonas/Programs/ALF/Prog/ALF.out
 ALF Copyright (C) 2016 - 2022 The ALF project contributors
 This Program comes with ABSOLUTELY NO WARRANTY; for details see license.GPL
 This is free software, and you are welcome to redistribute it under certain conditions.
 ALF Copyright (C) 2016 - 2022 The ALF project contributors
 This Program comes with ABSOLUTELY NO WARRANTY; for details see license.GPL
 This is free software, and you are welcome to redistribute it under certain conditions.
 ALF Copyright (C) 2016 - 2022 The ALF project contributors
 This Program comes with ABSOLUTELY NO WARRANTY; for details see license.GPL
 This is free software, and you are welcome to redistribute it under certain conditions.
 ALF Copyright (C) 2016 - 2022 The ALF project contributors
 This Program comes with ABSOLUTELY NO WARRANTY; fo

HDF5-DIAG: Error detected in HDF5 (1.14.4-3) thread 0:
  #000: ../hdf5-1.14.4-3/src/H5F.c line 836 in H5Fopen(): unable to synchronously open file
    major: File accessibility
    minor: Unable to open file
  #001: ../hdf5-1.14.4-3/src/H5F.c line 796 in H5F__open_api_common(): unable to open file
    major: File accessibility
    minor: Unable to open file
  #002: ../hdf5-1.14.4-3/src/H5VLcallback.c line 3863 in H5VL_file_open(): open failed
    major: Virtual Object Layer
    minor: Can't open object
  #003: ../hdf5-1.14.4-3/src/H5VLcallback.c line 3675 in H5VL__file_open(): open failed
    major: Virtual Object Layer
    minor: Can't open object
  #004: ../hdf5-1.14.4-3/src/H5VLnative_file.c line 128 in H5VL__native_file_open(): unable to open file
    major: File accessibility
    minor: Unable to open file
  #005: ../hdf5-1.14.4-3/src/H5Fint.c line 1960 in H5F_open(): unable to lock the file
    major: File accessibility
    minor: Unable to lock file
  #006: ../hdf5-1.14.4-3/src/

[mpiexec@DenkBrettA] Sending Ctrl-C to processes as requested
[mpiexec@DenkBrettA] Press Ctrl-C again to force abort

=   BAD TERMINATION OF ONE OF YOUR APPLICATION PROCESSES
=   RANK 0 PID 28924 RUNNING AT DenkBrettA
=   KILLED BY SIGNAL: 6 (Aborted)

=   BAD TERMINATION OF ONE OF YOUR APPLICATION PROCESSES
=   RANK 1 PID 28925 RUNNING AT DenkBrettA
=   KILLED BY SIGNAL: 2 (Interrupt)

=   BAD TERMINATION OF ONE OF YOUR APPLICATION PROCESSES
=   RANK 2 PID 28926 RUNNING AT DenkBrettA
=   KILLED BY SIGNAL: 6 (Aborted)

=   BAD TERMINATION OF ONE OF YOUR APPLICATION PROCESSES
=   RANK 3 PID 28927 RUNNING AT DenkBrettA
=   KILLED BY SIGNAL: 6 (Aborted)

=   BAD TERMINATION OF ONE OF YOUR APPLICATION PROCESSES
=   RANK 4 PID 28928 RUNNING AT DenkBrettA
=   KILLED BY SIGNAL: 6 (Aborted)

=   BAD TERMINATION OF ONE OF YOUR APPLICATION PROCESSES
=   RANK 5 PID 28929 RUNNING AT DenkBrettA
=   KILLED BY SIGNAL: 6 (Aborted)

=   BAD TERMINATION OF ONE OF YOUR APPLICATION PROCESSES
=   RANK 6 PI

KeyboardInterrupt: 

In [ ]:
sim.print_info_file()

In [9]:
sim.check_warmup(['m_scal', 'ising_x_scal'], gui='ipy')

In [10]:
sim.check_rebin(['m_scal', 'ising_x_scal'], gui='ipy', custom_obs={})

## Run a series of simulations
Create a list of `Simulation` instances with varying transverse Ising field $h$.

MPI parallelization is enabled, with number of processes set to 4. This results in 4 quasi-independent runs with the same parameters and therefore more precise outcomes.

In [17]:
L = 4
simsL4 = [
    Simulation(
        alf_src,
        'Nematic_Dirac_demo',
        {
            # Model specific parameters
            'L1': L,
            'L2': L,
            'beta': L*4.,
            'Ham_xi': 0.25,
            'Ham_h': h,
            # QMC parameters
            'Ltau': 1,
            'CPU_MAX': 1.,
            'NSweep': 20,
        },
        machine='gnu',
        mpi=True,
        n_mpi=8,
    )
    for h in [2.5, 3.0, 3.5, 4.0]]

In [10]:
simsL4

In [11]:
# Simple example for Pythons list comprehension (and f-strings)
[f'h={h}' for h in [2.5, 3.0, 3.5, 4.0]]

['h=2.5', 'h=3.0', 'h=3.5', 'h=4.0']

The simulation directories are:

In [11]:
for sim in simsL4:
    print(sim.get_directories())

['/home/jovyan/alf_workshop2020/Presentations/ALF_2024-advanced_pyALF/ALF_data/Nematic_Dirac_demo_L1=4_L2=4_beta=16.0_xi=0.25_h=2.5']
['/home/jovyan/alf_workshop2020/Presentations/ALF_2024-advanced_pyALF/ALF_data/Nematic_Dirac_demo_L1=4_L2=4_beta=16.0_xi=0.25_h=3.0']
['/home/jovyan/alf_workshop2020/Presentations/ALF_2024-advanced_pyALF/ALF_data/Nematic_Dirac_demo_L1=4_L2=4_beta=16.0_xi=0.25_h=3.5']
['/home/jovyan/alf_workshop2020/Presentations/ALF_2024-advanced_pyALF/ALF_data/Nematic_Dirac_demo_L1=4_L2=4_beta=16.0_xi=0.25_h=4.0']


This behaviour can be overwritten through the optional arguments `sim_dir` and `sim_root`.

Re-compiling ALF from the previous run is not necessary, since no compile-time parameters were changes.

Run the list of simulations in sequence.

In [ ]:
for sim in simsL4:
    sim.run()

Prepare directory "/home/jovyan/alf_workshop2020/Presentations/ALF_2024-advanced_pyALF/ALF_data/Nematic_Dirac_demo_L1=4_L2=4_beta=16.0_xi=0.25_h=2.5" for Monte Carlo run.
Create new directory.
Run /var/lib/ALF/Prog/ALF.out
 ALF Copyright (C) 2016 - 2022 The ALF project contributors
 This Program comes with ABSOLUTELY NO WARRANTY; for details see license.GPL
 This is free software, and you are welcome to redistribute it under certain conditions.
 No initial configuration


Check info files produced by ALF. It is strongly advised to take a look at these after finished runs, in particular "Precision Green" and "Precision Phase". These should be small, of order $10^{-8}$ or smaller. If they're bigger, one should decrease the stabilization invervall `Nwrap` (see parameter list `'VAR_QMC'` above). In our case, they're of order $10^{-14}$ and one might consider increasing `Nwrap` to speed up the simulation.

In [18]:
for sim in simsL4:
    sim.print_info_file()

===== /home/jovyan/alf_workshop2020/Presentations/ALF_2024-advanced_pyALF/ALF_data/Nematic_Dirac_demo_L1=4_L2=4_beta=16.0_xi=0.25_h=2.5/info =====
 Model is            : Nematic_Dirac
 L1                  :            4
 L2                  :            4
 N_SUN               :            2
 ham_t               :    1.0000000000000000     
 dtau                :   0.10000000000000001     
 beta                :    16.000000000000000     
 Ham_h               :    2.5000000000000000     
 Ham_J               :    1.0000000000000000     
 Ham_xi              :   0.25000000000000000     
 Ham_chem            :    0.0000000000000000     
 No initial configuration, Seed_in      402600
 Sweeps                              :           20
 Prog will stop after hours:    1.0000
 Measure Int.                        :            1         160
 Stabilization,Wrap                  :           10
 Nstm                                :           16
 Ltau                                :            1


## Running multiple parameters in parallel
The simulation object below shows an example of how use ALF's feature for running simulations for multiple parameters in parallel.
It is very similar to Parallel Tempering, but there are no exchange steps and the simulations are decoupled.

Parallel Tempering is enabled by making `sim_dict` a list of dictionaries, instead of a single dictionary. To instead use "Parallel Parameters", the argument `parallel_params=True` is necessary.

In [20]:
L = 6
simL6 = Simulation(
    alf_src,
    'Nematic_Dirac_demo',
    [{
        # Model specific parameters
        'L1': L,
        'L2': L,
        'beta': L*4.,
        'Ham_xi': 0.25,
        'Ham_h': h,
        # QMC parameters
        'Ltau': 1,
        'CPU_MAX': 4.,
        'NSweep': 20,
        'mpi_per_parameter_set': 8,
        'Tempering_calc_det': False,
    } for h in [2.5, 3.0, 3.5, 4.0]],
    machine='gnu',
    parallel_params=True,
    mpi=True,
    n_mpi=32,
)

In [14]:
simL6.get_directories()

['/home/jovyan/alf_workshop2020/Presentations/ALF_2024-advanced_pyALF/ALF_data/temper_Nematic_Dirac_demo_L1=6_L2=6_beta=24.0_xi=0.25_h=2.5/Temp_0',
 '/home/jovyan/alf_workshop2020/Presentations/ALF_2024-advanced_pyALF/ALF_data/temper_Nematic_Dirac_demo_L1=6_L2=6_beta=24.0_xi=0.25_h=2.5/Temp_1',
 '/home/jovyan/alf_workshop2020/Presentations/ALF_2024-advanced_pyALF/ALF_data/temper_Nematic_Dirac_demo_L1=6_L2=6_beta=24.0_xi=0.25_h=2.5/Temp_2',
 '/home/jovyan/alf_workshop2020/Presentations/ALF_2024-advanced_pyALF/ALF_data/temper_Nematic_Dirac_demo_L1=6_L2=6_beta=24.0_xi=0.25_h=2.5/Temp_3']

Recompilation from the previous normal MPI simulation is necessary!

In [15]:
simL6.compile()

Checking out branch Nematic_Dirac_demo
Your branch is up to date with 'origin/Nematic_Dirac_demo'.
Compiling ALF... 
Cleaning up Prog/


Already on 'Nematic_Dirac_demo'


Cleaning up Libraries/
Cleaning up Analysis/
Compiling Libraries


ar: creating modules_90.a
ar: creating libqrref.a


Compiling Analysis
Compiling Program
Parsing Hamiltonian parameters
filenames: Hamiltonians/Hamiltonian_Kondo_smod.F90 Hamiltonians/Hamiltonian_Kondo_read_write_parameters.F90
filenames: Hamiltonians/Hamiltonian_Hubbard_smod.F90 Hamiltonians/Hamiltonian_Hubbard_read_write_parameters.F90
filenames: Hamiltonians/Hamiltonian_Hubbard_Plain_Vanilla_smod.F90 Hamiltonians/Hamiltonian_Hubbard_Plain_Vanilla_read_write_parameters.F90
filenames: Hamiltonians/Hamiltonian_tV_smod.F90 Hamiltonians/Hamiltonian_tV_read_write_parameters.F90
filenames: Hamiltonians/Hamiltonian_LRC_smod.F90 Hamiltonians/Hamiltonian_LRC_read_write_parameters.F90
filenames: Hamiltonians/Hamiltonian_Z2_Matter_smod.F90 Hamiltonians/Hamiltonian_Z2_Matter_read_write_parameters.F90
filenames: Hamiltonians/Hamiltonian_Spin_Peierls_smod.F90 Hamiltonians/Hamiltonian_Spin_Peierls_read_write_parameters.F90
filenames: Hamiltonians/Hamiltonian_Nematic_Dirac_demo_smod.F90 Hamiltonians/Hamiltonian_Nematic_Dirac_demo_read_write_parameter

Here, we do not execute ALF, but only prepare the simulation.

In [52]:
simL6.run(copy_bin=True, only_prep=True)

Prepare directory "/home/jovyan/alf_workshop2020/Presentations/ALF_2024-advanced_pyALF/ALF_data/temper_Nematic_Dirac_demo_L1=6_L2=6_beta=24.0_xi=0.25_h=2.5" for Monte Carlo run.
Create new directory.
Prepare directory "/home/jovyan/alf_workshop2020/Presentations/ALF_2024-advanced_pyALF/ALF_data/temper_Nematic_Dirac_demo_L1=6_L2=6_beta=24.0_xi=0.25_h=2.5/Temp_0" for Monte Carlo run.
Create new directory.
Prepare directory "/home/jovyan/alf_workshop2020/Presentations/ALF_2024-advanced_pyALF/ALF_data/temper_Nematic_Dirac_demo_L1=6_L2=6_beta=24.0_xi=0.25_h=2.5/Temp_1" for Monte Carlo run.
Create new directory.
Prepare directory "/home/jovyan/alf_workshop2020/Presentations/ALF_2024-advanced_pyALF/ALF_data/temper_Nematic_Dirac_demo_L1=6_L2=6_beta=24.0_xi=0.25_h=2.5/Temp_2" for Monte Carlo run.
Create new directory.
Prepare directory "/home/jovyan/alf_workshop2020/Presentations/ALF_2024-advanced_pyALF/ALF_data/temper_Nematic_Dirac_demo_L1=6_L2=6_beta=24.0_xi=0.25_h=2.5/Temp_3" for Monte Carlo

In [21]:
simL6.print_info_file()

===== /home/jovyan/alf_workshop2020/Presentations/ALF_2024-advanced_pyALF/ALF_data/temper_Nematic_Dirac_demo_L1=6_L2=6_beta=24.0_xi=0.25_h=2.5/Temp_0/info =====
 Model is            : Nematic_Dirac
 L1                  :            6
 L2                  :            6
 N_SUN               :            2
 ham_t               :    1.0000000000000000     
 dtau                :   0.10000000000000001     
 beta                :    24.000000000000000     
 Ham_h               :    2.5000000000000000     
 Ham_J               :    1.0000000000000000     
 Ham_xi              :   0.25000000000000000     
 Ham_chem            :    0.0000000000000000     
 No initial configuration, Seed_in      604885
 Sweeps                              :           20
 Prog will stop after hours:    4.0000
 Measure Int.                        :            1         240
 Stabilization,Wrap                  :           10
 Nstm                                :           24
 Ltau                                :

In [22]:
L = 8
simL8 = Simulation(
    alf_src,
    'Nematic_Dirac_demo',
    [{
        # Model specific parameters
        'L1': L,
        'L2': L,
        'beta': L*4.,
        'Ham_xi': 0.25,
        'Ham_h': h,
        # QMC parameters
        'Ltau': 1,
        'CPU_MAX': 7.,
        'NSweep': 20,
        'Nbin': 1,
        'mpi_per_parameter_set': 8,
        'Tempering_calc_det': False,
    } for h in [2.5, 3.0, 3.5, 4.0]],
    machine='intel',
    parallel_params=True,
    mpi=True,
    n_mpi=32,
)

In [17]:
simL8.run(copy_bin=True, only_prep=True)

Prepare directory "/home/jovyan/alf_workshop2020/Presentations/ALF_2024-advanced_pyALF/ALF_data/temper_Nematic_Dirac_demo_L1=8_L2=8_beta=32.0_xi=0.25_h=2.5" for Monte Carlo run.
Prepare directory "/home/jovyan/alf_workshop2020/Presentations/ALF_2024-advanced_pyALF/ALF_data/temper_Nematic_Dirac_demo_L1=8_L2=8_beta=32.0_xi=0.25_h=2.5/Temp_0" for Monte Carlo run.
Resuming previous run.
Prepare directory "/home/jovyan/alf_workshop2020/Presentations/ALF_2024-advanced_pyALF/ALF_data/temper_Nematic_Dirac_demo_L1=8_L2=8_beta=32.0_xi=0.25_h=2.5/Temp_1" for Monte Carlo run.
Resuming previous run.
Prepare directory "/home/jovyan/alf_workshop2020/Presentations/ALF_2024-advanced_pyALF/ALF_data/temper_Nematic_Dirac_demo_L1=8_L2=8_beta=32.0_xi=0.25_h=2.5/Temp_2" for Monte Carlo run.
Resuming previous run.
Prepare directory "/home/jovyan/alf_workshop2020/Presentations/ALF_2024-advanced_pyALF/ALF_data/temper_Nematic_Dirac_demo_L1=8_L2=8_beta=32.0_xi=0.25_h=2.5/Temp_3" for Monte Carlo run.
Resuming prev

In [23]:
simL8.print_info_file()

===== /home/jovyan/alf_workshop2020/Presentations/ALF_2024-advanced_pyALF/ALF_data/temper_Nematic_Dirac_demo_L1=8_L2=8_beta=32.0_xi=0.25_h=2.5/Temp_0/info =====
 Model is            : Nematic_Dirac
 L1                  :            8
 L2                  :            8
 N_SUN               :            2
 ham_t               :    1.0000000000000000     
 dtau                :   0.10000000000000001     
 beta                :    32.000000000000000     
 Ham_h               :    2.5000000000000000     
 Ham_J               :    1.0000000000000000     
 Ham_xi              :   0.25000000000000000     
 Ham_chem            :    0.0000000000000000     
 No initial configuration, Seed_in      604885
 Sweeps                              :           20
 Prog will stop after hours:   10.0000
 Measure Int.                        :            1         320
 Stabilization,Wrap                  :           10
 Nstm                                :           32
 Ltau                                :

In [24]:
L = 10
simL10 = Simulation(
    alf_src,
    'Nematic_Dirac_demo',
    [{
        # Model specific parameters
        'L1': L,
        'L2': L,
        'beta': L*4.,
        'Ham_xi': 0.25,
        'Ham_h': h,
        # QMC parameters
        'Ltau': 1,
        'CPU_MAX': 7.,
        'NSweep': 20,
        'mpi_per_parameter_set': 8,
    } for h in [2.5, 3.0, 3.5, 4.0]],
    machine='gnu',
    parallel_params=True,
    mpi=True,
    n_mpi=32,
    #mpiexec='orterun',
)

In [ ]:
simL10.run()

In [23]:
simL10.print_info_file()

===== /home/jovyan/alf_workshop2020/Presentations/ALF_2024-advanced_pyALF/ALF_data/temper_Nematic_Dirac_demo_L1=10_L2=10_beta=40.0_xi=0.25_h=2.5/Temp_0/info =====
 Model is            : Nematic_Dirac
 L1                  :           10
 L2                  :           10
 N_SUN               :            2
 ham_t               :    1.0000000000000000     
 dtau                :   0.10000000000000001     
 beta                :    40.000000000000000     
 Ham_h               :    2.5000000000000000     
 Ham_J               :    1.0000000000000000     
 Ham_xi              :   0.25000000000000000     
 Ham_chem            :    0.0000000000000000     
 No initial configuration, Seed_in      604885
 Sweeps                              :           20
 Prog will stop after hours:   16.0000
 Measure Int.                        :            1         400
 Stabilization,Wrap                  :           10
 Nstm                                :           40
 Ltau                               